# Assignment APIs tutorial

In this notebook we are using the python package "millionaire-client" to interact with the deployed application for the NLP assignment 2026.

Required files:
- Directory called "millionaire_client"
- 1 colab notebook

Both files must be saved in a directory in your Google Drive, for example:
```
gDrive_home/
├── Colab Notebooks/
│   └── NLP_assignment/
│       ├── PoliMillionaire.ipynb <-- Your notebook
│       └── millionaire_client/ <-- Directory provided
```

### Sign up procedure
Before showing you how the api work, you need to signup from a web browser.
- Paste this link into your browser [http://131.175.15.22:51111/](http://131.175.15.22:51111/) this is where the demo is deployed
- You will see a standard login/sign up screen, please click on sign up
- In the "email" field please enter your politecnico email, you are allwed to create only 1 account using the same email you registered to the NLP course
- Choose whever username/password you prefer (be creative ;))

### Game interaction

Once you signed up, you can start interacting already from the api.

First of all, let's connect your drive to this Colab Notebook

In [ ]:
from google.colab import drive
import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, TextStreamer
import os

drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

def generate_outputs(system_message, user_message, stream=False):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    outputs = pipe(
        messages,
        max_new_tokens=600,
        do_sample=True,
        temperature=0.3,
        return_full_text=False,
        streamer = TextStreamer(tokenizer, skip_prompt = False) if stream else None
    )
    response = outputs[0]['generated_text'][-1]['content']
    return response




config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Then we need to add our python package "millionaire_client" to the system path, so python can see it.

In [ ]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment']


Let's import the client classes

In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

You can save your password in a Colab secret (the "key" icon on the tab on the left) and import it into your notebook.

In [ ]:
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

Now keep the API_URL as stated, but please change the username and password to be the ones you used during sign up session.

In [ ]:
API_URL = "http://131.175.15.22:51111/"
username = "riccardo"
password = pwd

Now we can instantiate a MillionaireClient object and call the login method, which takes as parameters username and password.

In [ ]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, stelo! (Role: student)


After login, the web page is showing you different types of competitions, for each of them you can choose to play a game or to see the leaderboard. For now let's list all of the.

In [ ]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)


In [ ]:
# Choose a competition ID
comp_id = 1

After choosing a competition, we can start a game! We can choose to start a game by calling `game = client.game.start(competition_id=comp_id)`. The object game is the one that is handling the game itself, we can call:
- game.current_question.text : to know the current question in text format
- game.current_level: to check the current level of difficulty of the question
- game.current_question.options: to check the possible choices we have to answer the question
- game.answer: to send to the server the answer we choose (the integer corresponding to our choice) and get the response (either correct or incorrect)

WATCH OUT! Each question has a timer, you have maximum 30 seconds to answer the question. As of now, if you exceed the maximum allowed time, there is not a "push notification". You still have to submit your answer anyway and, even though the answer was correct, you will get a TimedOut response!

In [ ]:
def play_game(game):
  # Play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      # Get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      # Get answer
      try:
          text = game.current_question.text

          output = generate_outputs(
              system_message="Reply with a single string that follows this structure: X_Whole-response, where X is the number of the option.",
              user_message=f"```{text}```",
          )
          print("answer: " + output)
          answer_id = int(output[0])
      except ValueError:
          print("Invalid input. Please enter a number.")
          continue

      # Submit answer
      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [ ]:
# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")
print()
play_game(game)


=== Starting Game ===


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Session ID: 8648
Total number of questions: 15


--- Level 1 ---
Q: What is the connection between Julius Caesar and the modern calendar?

  [0] He created the first 12-month calendar
  [1] He renamed the month Quintilis to July
  [2] He introduced the concept of leap years
  [3] He established the 7-day week

Time remaining: 29.9s


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


answer: 1, The Roman calendar was based on the Egyptian calendar, which was in turn influenced by the Mayan calendar.
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
Q: Which of the following is a key distinction between the pronunciation of Classical Attic and Koine Greek?

  [0] Classical Attic had more consonants than Koine Greek
  [1] Classical Attic had a three-way distinction between /b/, /p/, and /pʰ/
  [2] Koine Greek had a different system of accentuation
  [3] Koine Greek had more diphthongs than Classical Attic

Time remaining: 29.9s


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


answer: 1

 CORRECT!
 Earned so far: $200.00

--- Level 3 ---
Q: Which of the following best describes the fundamental principle of Roman city planning?

  [0] Organized around a central park
  [1] Based on a grid system with a central forum
  [2] Laid out randomly to confuse invaders
  [3] Designed for maximum density

Time remaining: 29.9s


Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


answer: 1, The city’s layout is designed to maximize the use of the land and facilitate efficient water management.
 CORRECT!
 Earned so far: $300.00

--- Level 4 ---
Q: Which of the following best describes the Temple of Olympian Zeus in Athens?

  [0] It was the largest temple in the ancient world during its completion.
  [1] It was primarily dedicated to the god Poseidon.
  [2] It was never used after its completion.
  [3] It was completed in the 6th century BC.

Time remaining: 29.9s
answer: 1, The Temple of Olympian Zeus was a massive temple dedicated to Zeus, the king of the gods, and was the largest free-standing structure in the ancient world.
 WRONG ANSWER!

 Game Over!
 Final earnings: $300.00

=== Game Summary ===
Reached Level: 4
Total Earnings: $300.00


In [ ]:
# Show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")


=== Leaderboard for Ancient History and Politics ===
  1. mark: $200.00 (Level 2)
  2. nicolo: $100.00 (Level 1) <-- YOU
  3. ftoschi: $100.00 (Level 1)
